# Visualization 2 - Regional effect

In this Notebook, we will be be working with a data set of baby names in France. We will try to answer the following questions about the data :
- Is there a regional effect in the data? 
- Are some names more popular in some regions? 
- Are popular names generally popular across the whole country?

In [ ]:
## Import libraries
import pandas as pd
import altair as alt
import json


# This will use an external file to store our data instead of embedding it directly in the
# visualization
alt.data_transformers.enable('json')

pass # Don't show any output in this cell

Let's import and take a look at the data!

In [ ]:
data = pd.read_csv('dpt2020.csv', sep=';')
#data = data.sample(frac=0.1)
data['dpt'] = data['dpt'].astype(str)

data.head()

Let's delete the "*_PRENOMS_RARES*" lines.

In [ ]:
data = data[data["preusuel"] != "_PRENOMS_RARES"]

Let's import the geospatial data.

In [ ]:
with open("departements.geojson", "r", encoding="utf-8") as f:
    geojson = json.load(f)


df_geo = pd.DataFrame([
    {
        "properties.code": feature["properties"]["code"],
        "properties.dep_name": feature["properties"]["nom"],
        "type": feature["type"],
        "geometry": feature["geometry"] # Dictionnaire GeoJSON pur préservé
    }
    for feature in geojson["features"]])

df_geo['properties.code'] = df_geo['properties.code'].astype(str)
df_geo[(df_geo['properties.code'] != '2A') & (df_geo['properties.code'] != '2B')]

df_geo.head(5) 

Input (selected year)

In [ ]:
year_input = alt.binding_select(
    options=[x for x in range(1900, 2021)], 
    name='Year: '
)

year_param = alt.param(
    name="selected_year",
    value=1900,
    bind=year_input
)

''' 
year_slider = alt.binding_range(
    min=1900,
    max=2020,
    step=1,
    name="select_region a year: "
)

year_param = alt.param(
    name="selected_year",
    value=1900,
    bind=year_slider
)
'''

We create a base chart, fusing both the baby names dataset and the geospatial data.

In [ ]:
highlight_region = alt.selection_point(name="highlight_region", on="pointerover", empty=False)
select_region = alt.selection_point(name="select_region", on="click", fields=['dpt'])

base_chart = alt.Chart(data).add_params(
    year_param, highlight_region, select_region
).transform_filter(
    "datum.annais == selected_year"
).transform_window(
    rank='rank()',
    sort=[alt.SortField('nombre', order='descending')],
    groupby=['dpt']
).transform_filter(
    "datum.rank == 1"
).transform_lookup(
    lookup='dpt',
    from_=alt.LookupData(
        data=df_geo,
        key='properties\.code', 
        fields=['geometry', 'type', 'properties\.dep_name'],
    )
).mark_geoshape(
    strokeWidth=0.3,
    stroke="black",
).encode(
    color=alt.condition(
        select_region, 
        alt.Color("preusuel:N", title="Top Name", scale=alt.Scale(scheme='category20')),
        alt.value("white"), 
    ),
    opacity=alt.condition(highlight_region, alt.value(0.6), alt.value(1)),
    tooltip=[
        alt.Tooltip("properties\.dep_name:N", title="Department"),
        alt.Tooltip("preusuel:N", title="Most common name"),
        alt.Tooltip("nombre:Q", title="Nb of occurrences")
    ]
).properties(
    width=500, 
    height=400,
    title=alt.TitleParams(
        text=alt.expr("'Most given baby names in France, by department, in ' + selected_year")
    ),
).project(
    type='mercator'
)

Then a subchart, showing the most popular baby names in the selected department.

In [ ]:
select_name = alt.selection_point(name="select_name", on="click", fields=['preusuel'], empty=False)
highlight_name = alt.selection_point(name="highlight_name", on="pointerover", empty=False)

by_dep_chart = alt.Chart(data).add_params(
    year_param,
    select_name,
    highlight_name
).transform_filter(
    "datum['annais'] == selected_year"
).transform_filter(
    "select_region.dpt"
).transform_filter(
    select_region
).transform_window(
    rank='rank()',
    sort=[alt.SortField('nombre', order='descending')]
).transform_filter(
    "datum.rank <= 20" 
).transform_lookup(
    lookup='dpt',
    from_=alt.LookupData(
        data=df_geo,
        key='properties\.code', 
        fields=['properties\.dep_name'],
    )
).mark_bar(
    size=20
).encode(
    x=alt.X(
        'preusuel:N',
        sort="-y", 
        title="Name",
        axis=alt.Axis(labelAngle=-45) 
    ),
    y=alt.Y('nombre:Q', title='Nb of occurrences'),
    opacity=alt.condition(highlight_name, alt.value(0.6), alt.value(1)),
    tooltip=[
        alt.Tooltip("properties\.dep_name:N", title="Department"),
        alt.Tooltip("preusuel:N", title="Most common name"),
        alt.Tooltip("nombre:Q", title="Nb of occurrences")
    ],
    color=alt.condition(
        select_name,
        alt.value('orange'),
        alt.value('blue')
    )
).properties(
    width=600,
    height=300,
    title=alt.TitleParams(
        text=alt.expr(
            "select_region.dpt ? "
            "'Most given baby names in ' + (select_region.dpt ? ('department n°' + select_region.dpt) : 'France') + ', in ' + selected_year :"
            "'Please select a department on the chart.'"
        ),
    )
)

Then a subchart for the subchart, showing the distribution of the selected name across France.

In [ ]:

name_map = alt.Chart(data).add_params(
    year_param, select_name
).transform_filter(
    "datum.annais == selected_year"
).transform_filter(
    select_name
).transform_lookup(
    lookup='dpt',
    from_=alt.LookupData(
        data=df_geo,
        key='properties\.code', 
        fields=['geometry', 'type', 'properties\.dep_name'],
    )
).mark_geoshape(
    strokeWidth=0.3,
    stroke="black",
).encode(
    color=alt.condition(
        select_name,
        alt.Color("nombre:Q", title="Nb of occurrences"),
        alt.value('white')
    ),
    tooltip=[
        alt.Tooltip("properties\.dep_name:N", title="Department"),
        alt.Tooltip("nombre:Q", title="Nb of occurrences")
    ]
).properties(
    width=500, 
    height=400,
    title=alt.TitleParams(
        text=alt.expr(
            "select_name.preusuel ? "
            "'Number of babies named ' + select_name.preusuel + ', by department, in ' + selected_year : "
            "'Please select a name on the subchart.'"
        )
    )
).project(
    type='mercator'
)

(base_chart | by_dep_chart | name_map).configure_title(fontSize=18)